# PoliMillionaire baseline

Before running this notebook, put the files in Google Drive like this:

```
MyDrive/
└── Colab Notebooks/
    └── NLP_assignment/
        ├── poli_millionaire_clean_baseline_v2.ipynb
        └── millionaire_client/
            ├── __init__.py
            ├── client.py
            ├── auth.py
            ├── base.py
            ├── game.py
            ├── models.py
            ├── competitions.py
            ├── leaderboard.py
            └── exceptions.py
```

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import torch

Mounted at /content/gdrive/


In [4]:
BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('BASE_DIR not found. Create the NLP_assignment folder in Drive and upload the notebook there.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print('Path added successfully.')

BASE_DIR exists: True
BASE_DIR contents: ['test3_offline_rag_index.pkl', 'pdfs', '.ipynb_checkpoints', 'PoliMillionaire.ipynb', 'millionaire_client', '.DS_Store', 'test3_rag_game_runs']
PACKAGE_DIR exists: True
PACKAGE_DIR contents: ['exceptions.py', '__pycache__', 'game.py', 'leaderboard.py', 'models.py', '__init__.py', 'client.py', 'auth.py', 'competitions.py', 'base.py']
Path added successfully.


In [5]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00:00:0100:01


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

In [7]:
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'gary'
PASSWORD = '13790229'

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

Logged in as: gary


In [8]:
competitions = client.competitions.list_all()
for c in competitions:
    print(c.id, c.name, c.max_levels)

COMPETITION_ID = competitions[3].id

0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


In [9]:
import torch
free, total = torch.cuda.mem_get_info()
print("Free GB:", free / 1024**3)
print("Total GB:", total / 1024**3)
print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)

Free GB: 14.46075439453125
Total GB: 14.56317138671875
Allocated GB: 0.0
Reserved GB: 0.0


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
quant_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

print("Model loaded:", model_name)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


In [40]:
from transformers import StoppingCriteria, StoppingCriteriaList
import time
import re
import csv
import os
from datetime import datetime

# Toggle: set True to enable Chain-of-Thought reasoning (longer outputs),
# set False for fast single-letter answers.
COT_MODE = True

LOG_CSV = 'generation_log.csv'

# Prepare log file header if missing
if not os.path.exists(LOG_CSV):
    with open(LOG_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'cot', 'question', 'parsed_letter', 'elapsed_sec', 'raw_output'])


def log_result(question_text, cot_mode, parsed_letter, elapsed, raw):
    with open(LOG_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([datetime.utcnow().isoformat(), 'COT' if cot_mode else 'FAST', question_text, parsed_letter, f"{elapsed:.3f}", raw.replace('\n', '\\n')])


class TimeLimitCriteria(StoppingCriteria):
    def __init__(self, start_time, max_time_seconds):
        self.start_time = start_time
        self.max_time = max_time_seconds
        
    def __call__(self, input_ids, scores, **kwargs):
        # Stop generation if we exceed the time limit
        return time.time() - self.start_time > self.max_time


class AnswerStopCriteria(StoppingCriteria):
    def __init__(self, prompt_length, tokenizer, min_reason_tokens=12):
        self.prompt_length = prompt_length
        self.tokenizer = tokenizer
        self.min_reason_tokens = min_reason_tokens

    def __call__(self, input_ids, scores, **kwargs):
        generated_ids = input_ids[0, self.prompt_length:]
        gen_len = generated_ids.numel()
        if gen_len == 0:
            return False

        generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        if not generated_text:
            return False

        # Only accept a single-letter final answer after a short reasoning window
        last_token_match = re.search(r'([ABCD0-3])\s*$', generated_text, flags=re.IGNORECASE)
        if last_token_match and gen_len >= self.min_reason_tokens:
            # Ensure the letter appears on its own line or at end after a newline
            if re.search(r'(?:\n|^)[ \t]*[ABCD0-3][ \t]*$', generated_text, flags=re.IGNORECASE):
                return True
            # also accept if the entire generated_text is just the letter
            if re.fullmatch(r'[ABCD0-3][\s\.,;:\)\]]*', generated_text, flags=re.IGNORECASE):
                return True

        return False


# Precompile common regexes for performance
_FINAL_LETTER_PATTERN = re.compile(r'final\s*answer[:\s]*([ABCD0-3])', flags=re.IGNORECASE)
_BOXED_PATTERN = re.compile(r'\\?boxed\{([ABCD])\}', flags=re.IGNORECASE)
_PAREN_PATTERN = re.compile(r'\(([ABCD])\)', flags=re.IGNORECASE)
_STANDALONE_LETTER = re.compile(r'\b([ABCD])\b', flags=re.IGNORECASE)
_DIGIT_PATTERN = re.compile(r'\b([0-3])\b')


def read_final_answer(text):
    """Parse model output and return a single letter A/B/C/D if found, else None.

    Strategy:
      - Check explicit 'Final Answer' style markers (letters or digits)
      - Check boxed, parenthesis, standalone letters, digits
      - Prefer the last explicit match
      - Return None if nothing found
    """
    if not text:
        return None

    txt = text

    # Prefer explicit final answer markers (may include digits)
    matches = _FINAL_LETTER_PATTERN.findall(txt)
    if matches:
        m = matches[-1]
        if m.isdigit():
            return _map_digit_to_letter(m)
        else:
            return m.upper()

    # Check boxed
    matches = _BOXED_PATTERN.findall(txt)
    if matches:
        return matches[-1].upper()

    # Parenthesis
    matches = _PAREN_PATTERN.findall(txt)
    if matches:
        return matches[-1].upper()

    # Digits anywhere (0-3)
    digits = _DIGIT_PATTERN.findall(txt)
    if digits:
        return _map_digit_to_letter(digits[-1])

    # Standalone letter last resort
    letters = _STANDALONE_LETTER.findall(txt)
    if letters:
        return letters[-1].upper()

    return None


def _map_digit_to_letter(d):
    if d is None:
        return None
    d = str(d).strip()
    return {'0':'A','1':'B','2':'C','3':'D'}.get(d, None)


def extract_letter(text):
    # Return None on failure; caller will decide fallback
    return read_final_answer(text)


def choose_answer(question, cot_mode=COT_MODE):
    if len(question.options) < 4:
        return question.options[0].id, 'A', 'fallback'

    if cot_mode:
        prompt = f'''You are the fastest and the most accurate math solver. Think step-by-step and show your chain-of-thought. At the end, write 'Final Answer:' on its own line followed by a single letter A, B, C, or D.
Question: {question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Final Answer: '''
    else:
        prompt = f'''You are the fastest and the most accurate math solver. Think concisely and answer the multiple choice question with a single letter (A,B,C,D). Provide at most one brief sentence of justification, then on its own final line write 'Final Answer:' followed by exactly one letter (A,B,C, or D).
Question: {question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Final Answer: '''

    # Tokenize prompt with explicit padding and max_length to avoid silent truncation
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, padding=True, max_length=tokenizer.model_max_length).to(model.device)
    prompt_length = inputs['input_ids'].shape[1]

    # Start timer; set proactive cutoff at 25s
    start_time = time.time()
    proactive_cutoff = 25.0
    time_limit = TimeLimitCriteria(start_time, proactive_cutoff)

    if cot_mode:
        # Allow full time for chain-of-thought but stop at proactive_cutoff
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList([time_limit])
            )
    else:
        # Fast path: stop early when a short reasoning window ends with a letter, or at proactive_cutoff
        answer_limit = AnswerStopCriteria(prompt_length, tokenizer, min_reason_tokens=12)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList([answer_limit, time_limit])
            )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    time_spent = time.time() - start_time

    # Try to parse a letter from the initial generation
    letter = extract_letter(full_text)

    # If we hit the proactive cutoff and no letter, immediately resume with a strict wrap-up injection
    if letter is None and time_spent >= proactive_cutoff:
        print(f"Proactive wrap-up at ~{proactive_cutoff}s: asking model to finish and choose closest option immediately...")
        wrap_injection = "\n\nTime is nearly up. Wrap up now and output only the final answer letter (A, B, C or D) on a single final line.\nFinal Answer: "
        inj_inputs = tokenizer(wrap_injection, return_tensors='pt', add_special_tokens=False).to(model.device)
        resume_input_ids = torch.cat([inputs['input_ids'], inj_inputs['input_ids']], dim=1)
        resume_attention_mask = torch.ones_like(resume_input_ids)

        with torch.no_grad():
            wrap_outputs = model.generate(
                input_ids=resume_input_ids,
                attention_mask=resume_attention_mask,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList([AnswerStopCriteria(resume_input_ids.shape[1], tokenizer, min_reason_tokens=1)])
            )
        full_text = tokenizer.decode(wrap_outputs[0], skip_special_tokens=True)
        time_spent = time.time() - start_time
        letter = extract_letter(full_text)
        log_result(question.text, cot_mode, letter if letter else 'NONE', time_spent, full_text)

    # Log initial attempt if not already logged by wrap injection
    if time_spent < proactive_cutoff or (letter is not None):
        log_result(question.text, cot_mode, letter if letter else 'NONE', time_spent, full_text)

    # If parse failed, do up to 2 retries with stricter instruction (existing retry flow)
    retries = 0
    while letter is None and retries < 2:
        retries += 1
        print(f"Retry {retries}: parsing failed, trying strict injection...")
        if cot_mode:
            injection = "\n\nPlease finish and then print 'Final Answer:' followed by exactly one letter (A,B,C,D) on its own line:\nFinal Answer: "
        else:
            injection = "\n\nPlease output only the final answer letter (A, B, C or D) on a single line, nothing else:\nFinal Answer: "
        inj_inputs = tokenizer(injection, return_tensors='pt', add_special_tokens=False).to(model.device)
        resume_input_ids = torch.cat([inputs['input_ids'], inj_inputs['input_ids']], dim=1)
        resume_attention_mask = torch.ones_like(resume_input_ids)

        # For retry, allow a short finish window
        with torch.no_grad():
            final_outputs = model.generate(
                input_ids=resume_input_ids,
                attention_mask=resume_attention_mask,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList([AnswerStopCriteria(resume_input_ids.shape[1], tokenizer, min_reason_tokens=1)])
            )
        full_text = tokenizer.decode(final_outputs[0], skip_special_tokens=True)
        time_spent = time.time() - start_time
        letter = extract_letter(full_text)
        log_result(question.text, cot_mode, letter if letter else 'NONE', time_spent, full_text)

    # Final fallback: if still None, return option 0 but mark letter = None
    try:
        if letter is None:
            idx = 0
            letter_label = None
        else:
            idx = ['A','B','C','D'].index(letter)
            letter_label = letter
    except ValueError:
        idx = 0
        letter_label = None

    return question.options[idx].id, letter_label if letter_label is not None else 'NONE', full_text


# Quick unit tests for read_final_answer (sanity checks)
_samples = [
    "Final Answer: C",
    "Final Answer: \\boxed{C}",
    "The solution is... Final Answer: D",
    "A, B, C, D",
    "C",
    "Answer: (B)",
    "Some reasoning... \n\\boxed{B}",
    "Final Answer: 2",
    "Time's up. 3",
    "I think it's 0"
]
print('\n-- read_final_answer tests --')
for s in _samples:
    print('INPUT:', s)
    print('PARSED:', read_final_answer(s))
print('-- end tests --\n')


-- read_final_answer tests --
INPUT: Final Answer: C
PARSED: C
INPUT: Final Answer: \boxed{C}
PARSED: C
INPUT: The solution is... Final Answer: D
PARSED: D
INPUT: A, B, C, D
PARSED: D
INPUT: C
PARSED: C
INPUT: Answer: (B)
PARSED: B
INPUT: Some reasoning... 
\boxed{B}
PARSED: B
INPUT: Final Answer: 2
PARSED: C
INPUT: Time's up. 3
PARSED: D
INPUT: I think it's 0
PARSED: A
-- end tests --



In [41]:
def play_game():
    game = client.game.start(competition_id=COMPETITION_ID, mode='text')
    correct_count = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        print('Level:', game.current_level)
        print(question.text)
        for i, opt in enumerate(question.options):
            print(f"{chr(65+i)}) {opt.text}")

        option_id, letter, raw = choose_answer(question)
        print('Predicted:', letter, '| Raw output:', raw)

        try:
            result = game.answer(option_id)
        except TimeoutError:
            print('Timed out')
            break
        except RateLimitError:
            print('Rate limited, waiting...')
            time.sleep(5)
            result = game.answer(option_id)

        if result.correct:
            correct_count += 1

        print('Correct:', result.correct, '| Earned:', result.earned_amount)

        if result.game_over:
            break

        time.sleep(0.5)

    print(f'Final earned: {game.earned_amount} | Total correct answers: {correct_count}')

    return correct_count

In [46]:
play_game()

Level: 1
Which of the following statements is (are) true? I. In order to use a χ2 procedure, the expected value for each cell of a one- or two-way table must be at least 5. II. In order to use χ2 procedures, you must have at least 2 degrees of freedom. III. In a 4 × 2 two-way table, the number of degrees of freedom is 3.
A) I and II only
B) I and III only
C) I only
D) III only


/tmp/ipykernel_12787/2250345049.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  writer.writerow([datetime.utcnow().isoformat(), 'COT' if cot_mode else 'FAST', question_text, parsed_letter, f"{elapsed:.3f}", raw.replace('\n', '\\n')])


Predicted: D | Raw output: You are the fastest and the most accurate math solver. Think step-by-step and show your chain-of-thought. At the end, write 'Final Answer:' on its own line followed by a single letter A, B, C, or D.
Question: Which of the following statements is (are) true? I. In order to use a χ2 procedure, the expected value for each cell of a one- or two-way table must be at least 5. II. In order to use χ2 procedures, you must have at least 2 degrees of freedom. III. In a 4 × 2 two-way table, the number of degrees of freedom is 3.
A) I and II only
B) I and III only
C) I only
D) III only

Final Answer:  \boxed{D}
</think>

To determine which statements are true, we analyze each statement one by one:

I. In order to use a χ² procedure, the expected value for each cell of a one- or two-way table must be at least 5.

- This is true. The χ² test requires that the expected value for each cell is at least 5 to ensure the test's validity.

II. In order to use χ² procedures, you mu

0

In [42]:
max_correct = 0
correct_counts = []
for _ in range(5):
    correct_count = play_game()
    correct_counts.append(correct_count)
    if correct_count > max_correct:
        max_correct = correct_count
print(f'Max correct answers in 5 runs: {max_correct}') 

Level: 1
Suppose A and B are n x n invertible matrices, where n > 1, and I is the n x n identity matrix. If A and B are similar matrices, which of the following statements must be true?	I. A - 2I and B - 2I are similar matrices. II. A and B have the same trace.	 III. A^-1 and B^-1 are similar matrices.
A) II only
B) I, II, and III
C) I only
D) III only
Predicted: A | Raw output: You are the fastest and the most accurate math solver. Think concisely and answer the multiple choice "                 "question with a single digit (A,B,C,D).

You may provide a brief (one-line) justification, but end with the final answer on its own line. Reply with nothing else on that final line: A, B, C, or D.

Question: Suppose A and B are n x n invertible matrices, where n > 1, and I is the n x n identity matrix. If A and B are similar matrices, which of the following statements must be true?	I. A - 2I and B - 2I are similar matrices. II. A and B have the same trace.	 III. A^-1 and B^-1 are similar matr

In [ ]:
print(correct_counts)

In [ ]:
''' --- IGNORE ---
import gc
import torch

try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("CUDA VRAM emptied.")
''' 

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print("Free GB:", free / 1024**3)
print("Total GB:", total / 1024**3)
print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)